# RFM Customer Segmentation Analysis

This notebook analyzes customer recency, frequency, monetary value, and compares segment-level differences for CRM and retention strategy design.

## Project Background

- Data source table: `rfm_customer_segments`
- RFM scope: delivered orders only
- Customer grain: `customer_unique_id`

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 11


In [ ]:
def load_env_file(env_path: str = '../.env') -> None:
    env_file = Path(env_path)
    if not env_file.exists():
        return

    for line in env_file.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip())

load_env_file()

DB_HOST = os.getenv('DB_HOST', '127.0.0.1')
DB_PORT = os.getenv('DB_PORT', '3306')
DB_USER = os.getenv('DB_USER', 'root')
DB_PASSWORD = os.getenv('DB_PASSWORD', '')
DB_NAME = os.getenv('DB_NAME', 'ecommerce_gmv_customer_analysis')

engine = create_engine(
    f'mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4'
)

PROJECT_ROOT = Path('../').resolve()
DASHBOARD_DIR = PROJECT_ROOT / 'dashboards'
DASHBOARD_DIR.mkdir(parents=True, exist_ok=True)

print('Connected to database:', DB_NAME)
print('Dashboard directory:', DASHBOARD_DIR)


## Load RFM Result Table

In [ ]:
rfm_sql = """
SELECT
    customer_unique_id,
    recency,
    frequency,
    monetary,
    r_score,
    f_score,
    m_score,
    rfm_score,
    segment_name
FROM rfm_customer_segments;
"""

df_rfm = pd.read_sql(text(rfm_sql), engine)
df_rfm.head()


## Segment Distribution

In [ ]:
df_segment_count = (
    df_rfm['segment_name']
    .value_counts()
    .rename_axis('segment_name')
    .reset_index(name='customer_count')
)

df_segment_count


In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=df_segment_count, x='customer_count', y='segment_name')
plt.title('Customer Distribution by Segment')
plt.xlabel('Customer Count')
plt.ylabel('Segment Name')
plt.tight_layout()

output_path = DASHBOARD_DIR / 'rfm_segment_distribution.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
plt.show()

print('Saved:', output_path)


## Average Monetary by Segment

In [ ]:
df_segment_monetary = (
    df_rfm.groupby('segment_name', as_index=False)['monetary']
    .mean()
    .rename(columns={'monetary': 'avg_monetary'})
    .sort_values('avg_monetary', ascending=False)
)

df_segment_monetary


In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=df_segment_monetary, x='avg_monetary', y='segment_name')
plt.title('Average Monetary by Segment')
plt.xlabel('Average Monetary')
plt.ylabel('Segment Name')
plt.tight_layout()

output_path = DASHBOARD_DIR / 'rfm_avg_monetary_by_segment.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
plt.show()

print('Saved:', output_path)


## Average Frequency by Segment

In [ ]:
df_segment_frequency = (
    df_rfm.groupby('segment_name', as_index=False)['frequency']
    .mean()
    .rename(columns={'frequency': 'avg_frequency'})
    .sort_values('avg_frequency', ascending=False)
)

df_segment_frequency


In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=df_segment_frequency, x='avg_frequency', y='segment_name')
plt.title('Average Frequency by Segment')
plt.xlabel('Average Frequency')
plt.ylabel('Segment Name')
plt.tight_layout()
plt.show()


## Segment Contribution Analysis

In [ ]:
df_segment_contribution = (
    df_rfm.groupby('segment_name', as_index=False)
    .agg(
        customer_count=('customer_unique_id', 'count'),
        total_monetary=('monetary', 'sum'),
        avg_monetary=('monetary', 'mean')
    )
)

df_segment_contribution['monetary_pct'] = (
    df_segment_contribution['total_monetary']
    / df_segment_contribution['total_monetary'].sum()
    * 100
)

df_segment_contribution = df_segment_contribution.sort_values('total_monetary', ascending=False)
df_segment_contribution


In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=df_segment_contribution, x='total_monetary', y='segment_name')
plt.title('Total Monetary by Segment')
plt.xlabel('Total Monetary')
plt.ylabel('Segment Name')
plt.tight_layout()

output_path = DASHBOARD_DIR / 'rfm_total_monetary_by_segment.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
plt.show()

print('Saved:', output_path)


In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=df_segment_contribution, x='monetary_pct', y='segment_name')
plt.title('Monetary Contribution Percentage by Segment')
plt.xlabel('Monetary Contribution (%)')
plt.ylabel('Segment Name')
plt.tight_layout()

output_path = DASHBOARD_DIR / 'rfm_monetary_contribution_pct.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
plt.show()

print('Saved:', output_path)


## Distribution Analysis

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df_rfm['recency'], bins=30, kde=True)
plt.title('Recency Distribution')
plt.xlabel('Recency (Days)')
plt.ylabel('Customer Count')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df_rfm['frequency'], bins=20, kde=False)
plt.title('Frequency Distribution')
plt.xlabel('Frequency')
plt.ylabel('Customer Count')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df_rfm['monetary'], bins=30, kde=True)
plt.title('Monetary Distribution')
plt.xlabel('Monetary')
plt.ylabel('Customer Count')
plt.tight_layout()
plt.show()


In [ ]:
df_rfm_plot = df_rfm[df_rfm['monetary'] > 0].copy()
df_rfm_plot['log_monetary'] = np.log10(df_rfm_plot['monetary'])

plt.figure(figsize=(10, 5))
sns.histplot(df_rfm_plot['log_monetary'], bins=30, kde=True)
plt.title('Log Monetary Distribution')
plt.xlabel('log10(Monetary)')
plt.ylabel('Customer Count')
plt.tight_layout()
plt.show()


## Segment Summary Table

In [ ]:
df_segment_summary = (
    df_rfm.groupby('segment_name', as_index=False)
    .agg(
        customer_count=('customer_unique_id', 'count'),
        avg_recency=('recency', 'mean'),
        avg_frequency=('frequency', 'mean'),
        avg_monetary=('monetary', 'mean')
    )
    .sort_values('avg_monetary', ascending=False)
)

df_segment_summary


## Key Segment Insights

- High Value customers should combine stronger frequency and stronger monetary contribution.
- New Customers usually have better recency but limited purchase depth.
- At Risk customers may still hold meaningful historical value but show weaker recency.
- Segment contribution analysis reveals where total customer value is concentrated.

## CRM Recommendations

- Retain High Value customers with exclusive benefits and personalized care.
- Push second-purchase conversion for New Customers.
- Launch win-back campaigns for At Risk and Dormant customers.
- Use segment-level contribution results to guide CRM budget allocation and operational focus.